In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Quick inspection of a preprocessed BraTS subject (_img.h5 from generate_constraint_maps.py).
Reports stored attrs + per-channel data ranges, whole-image vs within-brain-mask.

    python inspect_brats_h5.py /path/to/BraTS2021_00000_img.h5
or import inspect_img(path) in a notebook.
"""

import sys
import numpy as np
import h5py


def inspect_img(img_h5_path, max_slices=None):
    with h5py.File(img_h5_path, "r") as f:
        img = np.asarray(f["img"][:max_slices] if max_slices else f["img"][:])   # (N,H,W,C)
        mask = np.asarray(f["mask"][:max_slices] if (max_slices and "mask" in f)
                          else f["mask"][:])[..., 0].astype(bool) if "mask" in f else None
        attrs = dict(f.attrs)

    N, H, W, C = img.shape
    contrasts = attrs.get("contrasts", "")
    names = contrasts.split(",") if isinstance(contrasts, str) else [str(c) for c in contrasts]
    print(f"file: {img_h5_path}")
    print(f"img: shape={img.shape} dtype={img.dtype}  contrasts={names}")
    print(f"attrs: normalize={attrs.get('normalize')} slice_range={attrs.get('slice_range')} "
          f"crop_size={attrs.get('crop_size')} n_clusters={attrs.get('n_clusters')}")
    if "norm_stats" in attrs:
        print(f"norm_stats (per channel, (mean,std) or (lo,range)):\n{np.asarray(attrs['norm_stats'])}")

    brain_frac = mask.reshape(N, -1).mean(1) if mask is not None else None
    if brain_frac is not None:
        print(f"brain fraction/slice: min={brain_frac.min():.3f} "
              f"mean={brain_frac.mean():.3f} max={brain_frac.max():.3f}")

    hdr = f"{'channel':<10}{'whole min/max':<22}{'whole mean/std':<22}"
    if mask is not None:
        hdr += f"{'brain min/max':<22}{'brain mean/std':<22}"
    print("\n" + hdr)
    for c in range(C):
        ch = img[..., c]
        line = (f"{(names[c] if c < len(names) else c):<10}"
                f"{f'{ch.min():.3f}/{ch.max():.3f}':<22}"
                f"{f'{ch.mean():.3f}/{ch.std():.3f}':<22}")
        if mask is not None:
            b = ch[mask]
            line += (f"{f'{b.min():.3f}/{b.max():.3f}':<22}"
                     f"{f'{b.mean():.3f}/{b.std():.3f}':<22}")
        print(line)

    if mask is not None:
        bg = img[~mask[..., None].repeat(C, -1).reshape(img.shape)]
        print(f"\nbackground voxels: all-zero={np.all(bg == 0)}  "
              f"min={bg.min():.4f} max={bg.max():.4f}  (expect exactly 0 with the sentinel)")
    return img, mask, attrs


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("usage: python inspect_brats_h5.py <subject>_img.h5")
    inspect_img(sys.argv[1])